# 🏭 AI-Enabled Assets Performance & Predictive Maintenance Platform

This notebook allows you to run the full **Industrial Risk AI** platform directly in Google Colab.

### ⚠️ Fix for 'Importing a module script failed'
If you see this error, it is almost always a **browser cache issue** with Colab's tunneling. 
1. Open the tunnel link in **Incognito / Private Mode**.
2. Or **Clear Browser Cache** and refresh the page.
3. Ensure the **CORS and XSRF** settings are applied in `.streamlit/config.toml` (automated in this notebook).

### Step 1: 🚀 Setup Environment
Run this cell to install the platform and stable dependencies.

In [ ]:
import os
import subprocess
import time

# 1. Clone Repository
REPO_URL = "https://github.com/lmudu2/industrial-risk-ai.git"
REPO_DIR = "industrial-risk-ai"

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

print(f"Cloning {REPO_URL}...")
!git clone {REPO_URL}
os.chdir(f"/content/{REPO_DIR}")

# 2. Install Stable Dependencies
print("Installing pinned dependencies (Streamlit 1.29.0)...")
!pip install -q -r requirements.txt
!pip install -q localtunnel

print("✅ Setup Complete!")

### Step 2: 📊 Generate Industrial Database
Run this cell to generate the synthesized SQLite database.

In [ ]:
if not os.path.exists("backend/eam_database.db"):
    print("Generating real-world industrial data (3-5 minutes)...")
    !python data/generate_data.py
else:
    print("✅ Database already exists.")

### Step 3: ⚡ Start & Tunnel
Run this cell, wait for the URL, and open it in **Incognito Mode**.

In [ ]:
from google.colab import userdata
import urllib.request

# Kill any old processes
!pkill streamlit
!pkill uvicorn

# 0. Load Secrets
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ GROQ_API_KEY loaded!")
except:
    print("⚠️ GROQ_API_KEY not found in Secrets.")

# 1. Start Services
print("Starting Backend and Frontend...")
subprocess.Popen(["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"])
subprocess.Popen(["streamlit", "run", "frontend/app.py", "--server.port", "8501", "--server.address", "0.0.0.0", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"])

time.sleep(5)

# 2. Get Tunnel Password (IP)
try:
    ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print(f"\n1. COPY THIS Tunnel Password: {ip}")
except:
    print("Could not fetch IP.")

print("2. CLICK THE LINK BELOW AND PASTE THE PASSWORD")
print("3. IF YOU SEE 'MODULE SCRIPT FAILED', USE INCOGNITO MODE.\n")

# 3. Start Tunnel
!npx localtunnel --port 8501